<a href="https://colab.research.google.com/github/luisferdm6/03-china-mexico-trade-audit/blob/main/01_auditoria_asimetrias_china_mexico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Auditoría de Asimetrías Comerciales y Riesgo Aduanal: China – México
### Análisis de datos espejo (Mirror Trade Statistics) para detección de brechas arancelarias y subvaluación

**Analista:** Luis Fernando Duardo Medina
**Metodología:** Estadísticas Espejo (UN Comtrade / OMA)  
**Stack Técnico:** DuckDB (SQL), Pandas, Seaborn, Matplotlib

In [5]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import ticker

In [6]:
df = pd.read_csv("comtrade_china_mexico.csv", index_col=False)

print(f"Dimensiones del dataset: {df.shape}")

df.head()

Dimensiones del dataset: (18, 47)


,typeCode,freqCode,refPeriodId,refYear,refMonth,period,reporterCode,reporterISO,reporterDesc,flowCode,...,netWgt,isNetWgtEstimated,grossWgt,isGrossWgtEstimated,cifvalue,fobvalue,primaryValue,legacyEstimationFlag,isReported,isAggregate
0,C,A,20230101,2023,52,2023,484,MEX,Mexico,M,...,0,True,0,False,NaN,765147704,765147704,4,False,True
1,C,A,20230101,2023,52,2023,484,MEX,Mexico,M,...,0,True,0,False,NaN,21701094738,21701094738,4,False,True
2,C,A,20230101,2023,52,2023,484,MEX,Mexico,M,...,0,True,0,False,NaN,39829628491,39829628491,4,False,True
3,C,A,20230101,2023,52,2023,484,MEX,Mexico,X,...,0,True,0,False,NaN,983645,983645,4,False,True
4,C,A,20230101,2023,52,2023,484,MEX,Mexico,X,...,0,True,0,False,NaN,551449072,551449072,4,False,True


In [7]:
df_limpio = df[['refYear', 'flowDesc', 'cmdCode', 'primaryValue']]

df_limpio.head()

,refYear,flowDesc,cmdCode,primaryValue
0,2023,Import,64,765147704
1,2023,Import,84,21701094738
2,2023,Import,85,39829628491
3,2023,Export,64,983645
4,2023,Export,84,551449072


In [11]:
consulta = duckdb.sql("""
SELECT
  flowDesc AS flujo,
  SUM(primaryValue) AS valor_total_usd
FROM df_limpio
GROUP BY flowDesc
""").df()

consulta

,flujo,valor_total_usd
0,Import,2.035462e+11
1,Export,4.515367e+09


In [13]:
consulta_capitulos = duckdb.sql("""
SELECT
  cmdCode AS capitulo,
  flowDesc AS flujo,
  SUM(primaryValue) AS total_usd
FROM df_limpio
GROUP BY cmdCode, flowDesc
ORDER BY cmdCode ASC, SUM(primaryValue) DESC
""").df()

consulta_capitulos

,capitulo,flujo,total_usd
0,64,Import,2.269195e+09
1,64,Export,1.708209e+06
2,84,Import,7.370048e+10
3,84,Export,1.506250e+09
4,85,Import,1.275765e+11
5,85,Export,3.007409e+09


In [18]:
consulta_balanza = duckdb.sql("""
SELECT
  cmdCode AS capitulo,
  SUM(CASE WHEN flowDesc = 'Import' THEN primaryValue ELSE 0 END) AS importaciones_usd,
  SUM(CASE WHEN flowDesc = 'Export' THEN primaryValue ELSE 0 END) AS exportaciones_usd,
  exportaciones_usd - importaciones_usd AS balanza_comercial_usd
FROM df_limpio
GROUP BY cmdCode
ORDER BY capitulo ASC
""").df()

consulta_balanza


,capitulo,importaciones_usd,exportaciones_usd,balanza_comercial_usd
0,64,2.269195e+09,1.708209e+06,-2.267487e+09
1,84,7.370048e+10,1.506250e+09,-7.219423e+10
2,85,1.275765e+11,3.007409e+09,-1.245691e+11
